# 0. 환경 설정

## 0-1. 라이브러리 설치

- RAG 파이프라인과 Agent 구현에 필요한 라이브러리 설치
1. LangChain 관련: `langchain-community`, `langchain-upstage` 등
2. Vector Stroe: `chromadb`
3. Document Loader: `pymupdf`, `pypdf`

In [ ]:
# LangChain 1.0+ 설치 (핵심, 커뮤니티, 파트너 패키지 포함)
# langchain: 핵심 + LCEL + Agent 등
# langchain-community: ChromaDB, TextLoader 등 커뮤니티 기반 연동
# langchain-upstage: Upstage LLM, Embedding 연동
# langchain-upstage가 구버젼과 호환성을 가지고 있어 -U를 통해 업그레이드 설치 권장
!pip install -U langchain 
!pip install -U langchain-core
!pip install -U langchain-community
!pip install -U langchain-upstage

# RAG 파이프라인 관련
!pip install langchain-text-splitters
!pip install tiktoken

# Vector Store
!pip install chromadb

# Document Loaders (PDF 처리용)
!pip install pypdf pymupdf pypdfium2

# 중요. 설치 완료 후 버젼 확인 필수

- 1.0 버젼이 올바르게 설치되었는지 확인
- 만약 구버젼이 확인된다면, `restart`
- `restart` 후에도 캐시 데이터에 의해 구버젼이 출력될 수도 있음.
- 만약, 실습 도중, 버젼 이슈가 발생한다면 다시 `restart`

In [ ]:
!pip show langchain
!pip show langchain-core

## 0-2. Upstage API Key 설정

- [Upstage API Docs](https://console.upstage.ai/docs/getting-started)
- dotenv 라이브러리 활용
    - **load_dotenv:** .env 파일에 저장된 KEY=VALUE 쌍을 환경 변수로 로드
    - **getenv:** 로드된 환경 변수 중 "UPSTAGE_API_KEY"에 해당하는 값을 읽음

In [ ]:
from google.colab import drive
from dotenv import load_dotenv
from os import getenv

# 1. Google Drive 마운트
drive.mount('/content/drive')

# 2. .env 파일이 저장된 경로 설정 (VIktor, 이 경로는 네 환경에 맞게 수정해야 해)
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/11_Agent/' 

In [ ]:
# (Colab에서 .env 파일이 없다면 이 코드로 생성)
# !mkdir -p "{base_path}"
!echo "UPSTAGE_API_KEY={Upstage_API}" > "{base_path}.env"

In [ ]:
# 3. .env 파일 로드
# base_path 경로에 있는 .env 파일을 찾아 환경 변수로 로드
# load_dotenv(".env")
load_dotenv(base_path + ".env")

# 4. 환경 변수 읽기
UPSTAGE_API_KEY = getenv("UPSTAGE_API_KEY")

if UPSTAGE_API_KEY:
    print("Success API Key Setting!")
else:
    # 키 로드에 실패하면 에러 메시지를 명확하게 표시
    print(f"ERROR: Failed to load UPSTAGE_API_KEY from .env")

## 0-3. DB 생성 및 데이터 삽입

- python으로 SQL 문법을 사용할 수 있도록 해주는 sqlite3 라이브러리 사용
- DB를 다루는 자세한 방법은 DB 수업 참고
1. Agent가 사용할 Tool이 처리할 기능을 위해 DB 생성
    - ex) 특정 `order_id`에 대한 정보 조회
    - 특정  `order_id`의 배송 상황에 따라 행동 양식 지정

In [ ]:
import sqlite3

# 1. DB 및 테이블 생성
def setup_refined_database():
    conn = sqlite3.connect(base_path + 'orders.db') # 파일 기반 DB 생성
    c = conn.cursor()
    try:
        # 테이블 구조
        c.execute('''
            CREATE TABLE orders (
                order_id TEXT PRIMARY KEY,
                customer_name TEXT,
                item_name TEXT,
                status TEXT
            )
        ''')
        # 데이터 추가
            # Delivered         : 배송 완료
            # Shipping Delayed  : 배송 지연
            # Processing        : 처리 중
        c.executemany('INSERT INTO orders (order_id, customer_name, item_name, status) VALUES (?, ?, ?, ?)', [
            ('ORDER123', 'Alice Smith', 'Introduction to Python', 'Delivered'),
            ('ORDER456', 'Bob Johnson', 'The AI Revolution', 'Shipping Delayed'),
            ('ORDER789', 'Charlie Lee', 'Data Structures', 'Processing'),
            ('ORDER101', 'David Kim', 'Learning SQL', 'Processing'),
            ('ORDER102', 'Alice Smith', 'Advanced ML', 'Delivered'),
            ('ORDER505', 'Eva Moon', 'Cloud Computing', 'Shipping Delayed') 
        ])
        conn.commit()   # 변경사항 저장
        print("--- Mock 'orders.db' 생성 완료 ---")
    except sqlite3.OperationalError:
        print("--- Mock 'orders.db'가 이미 존재합니다 ---")
    finally:
        conn.close()    # DB 연결 종료

# DB 셋업 실행
setup_refined_database()

# 1. RAG 및 Custom Tool

- Agent가 고객 민원을 해결하기 위해 필요한 두 종류의 도구 정의
1. **RAG**
    - 고객의 요구 사항은 `“정책이 어떻게 되나요?”` 와 같이, LLM이 바로 이해하기는 어려울 수 있는 형태의 질문일 가능성이 매우 높음.
    - 이때, LLM이 올바른 답변을 할 수 있도록 지식 검색 기능을 구축
2. **Custom Tool**
    - 지식 검색이 가능한 Vector Store를 구축하였다고 하더라도, LLM이 정확히 어떤 기능을 수행해야 하는지는 모호할 수 있음.
    - 각 상황에 적절한 대응이 가능하도록 Tool을 구성

## 1-1. 목표

1. yes24 정책을 VectorStore에서 관리
2. RAG 파이프라인을 구축하여, LLM이 적절한 응대가 가능하도록 설정
3. LLM이 답변에 활용할 각종 Tool 정의
4. 고객의 요구에 따라 LLM이 적절한 응답을 생성하도록 파이프라인 구축
5. 이를 위한 ReAct Agent 설꼐

## 1-2. RAG (retriever) 준비

### 1-2-1. 이전 실습 복기

- **collection:**
    - 벡터 스토어 내에서 벡터들을 그룹화하는 단위
    - 여러 컬렉션을 통해 다양한 주제나 도메인에 대한 벡터들을 효율적으로 관리 가능
    - ChromaDB에서는 하나의 데이터베이스 내에 여러 컬렉션을 생성하여 벡터들을 분류하고 관리할 수 있음
    - **예: "배송 정책" 컬렉션, "반품 정책" 컬렉션 등**

In [ ]:
import glob
# langchain-community에서 로더
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader 
# data chunking을 위한 텍스트 스플리터
from langchain_text_splitters import RecursiveCharacterTextSplitter
# langchain-community에서 Vector Store
from langchain_community.vectorstores import Chroma 
from langchain_upstage import UpstageEmbeddings

# 1. Load
# pdf_files = glob.glob('/data/*.pdf') # PDF 파일 경로
pdf_files = glob.glob(base_path + '/data/*.pdf') # PDF 파일 경로
documents = []
for pdf_filepath in pdf_files:
    loader = PyMuPDFLoader(pdf_filepath) # PyMuPDF로 PDF 로드
    documents.extend(loader.load())

# 배송 정책 TXT 파일 로드
# shipping_loader = TextLoader("shipping_policy.txt")
shipping_loader = TextLoader(base_path + "shipping_policy.txt")
documents.extend(shipping_loader.load())
print(f"총 {len(documents)}개의 문서를 로드했습니다.")

# 2. Chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, # 청크 크기
    chunk_overlap=20, # 중복 크기
)
chunks = text_splitter.split_documents(documents)
print(f"문서를 총 {len(chunks)}개의 청크로 분할했습니다.")

# 3. Embed & Store
embeddings = UpstageEmbeddings(model="embedding-query") # Upstage 임베딩 모델

# ChromaDB에 청크들을 임베딩하여 저장
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    # persist_directory="chroma_db_v1"
    persist_directory=base_path + "chroma_db_v1" 
)

print(f"Vector Store 생성 완료. 컬렉션 개수: {vectorstore._collection.count()}")

# 4. Retriever
retriever = vectorstore.as_retriever() # Vector Store를 검색기로 변환
print("RAG Retriever가 준비되었습니다.")

## 1-3. 기능 도구 정의

### 1-3-1. 신뢰성(Trustworthiness)

- AI 시스템에게 의사결정을 맡길 때, 가장 중요한 요소
1. 정확성
    - Agent가 제공하는 정보나 예측, 실행하는 작업이 사실과 부합하며, 오류가 없는 정도
2. 공정성
    - Agent가 특정 그룹이나 개인에게 편향되지 않고 공평하게 제공해야 함.
3. 설명 가능성
    - Agent가 내린 결정의 결정 과정과 이유를 사용자가 이해할 수 있도록 명확하게 설명 할 수 있어야 함.
4. 책무성
    - 작동에 문제가 발생했을 때, 그 책임을 물을 수 있는 주체와 대응 방안이 마련되어 있어야 함.
- 이번 실습에서는 DB에 저장되어 있는 정보와 `배송 정책` 문서를 그대로 참고할 것이므로 위 신뢰성 평가 항목들에 대해서는 가볍게 진행할 것.

### 1-3-2. 그럼에도 불구하고

1. 도구의 역할 및 범위 명확화
    - 도구가 무엇을 할 수 있는지를 명확히 하여야 함.
    1. 사용자가 배송 상태에 대해 물었을 때, `order_id`를 기반으로 DB를 조회하고, 응답할 수 있어야 함.
    2. 배송 지연 상황에 대해 적절한 대응이 가능하여야 함.
2. 오류 처리 및 예외 관리
    - 발생 할 수 있는 다양한 잠재적인 문제 (네트워크 오류, **잘못된 입력 형식**, DB 연결 실패 등)를 오류 상황에 대해 유연하게 대처 할 수 있어야 함.
3. 재현성
    - 도구가 일관되고 안전하게 결과를 제공할 수 있어야 함.
    - ex) 동일한 질문에 대해서 전혀 다른 대답을 해서는 안됨.

### 1-3-3. 도구 구현 및 등록
- 정의한 각 도구를 Agent가 활용 할 수 있도록 tool로 등록하는 과정 필요
- 이때, 각 도구가 어떠한 용도로 사용될 것인지 잘 작성해주는 것이 중요함.
- **주의**
    - 도구의 이름은 반드시 공백없이 `_` 또는 `-` 으로 구분하여야 함

1. RAG 파이프라인 (위에서 정의한 retriever)을 언제 활용 할 것인지 tool로 정의
    - create_retriever_tool를 사용하여 도구화

In [ ]:
from langchain_classic.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "Policy_Search",
    """배송 정책, 보상 정책, 환불 정책 등 AI 온라인 서점의 
    다양한 정책 문서에서 고객 질문과 관련된 정보를 검색합니다.
    (예: "배송 지연 시 보상 정책이 어떻게 되나요?")
    """,
)

In [ ]:
retriever_tool.invoke({"query": "주말에 배송 되나요?"})

2. 고객의 배송 조회에 대응 할 수 있는 tool 정의 (배송 DB 검색)
    - `@tool` 데코레이터를 활용하여 도구화

In [ ]:
from langchain.tools import tool

# @tool
# def search_database(query: str, limit: int = 10) -> str:
#     """Search the customer database for records matching the query.

#     Args:
#         query: Search terms to look for
#         limit: Maximum number of results to return
#     """
#     return f"Found {limit} results for '{query}'"

# 2. 고객 배송 상태 조회 도구 함수
@tool
def get_order_status_from_db(order_id: str) -> str:
    """(DB 조회) 
        주문 ID를 받아 현재 '배송 상태' 문자열을 반환
        단, DB 연결 오류 시 "DB Error: {오류 메시지}" 반환
        주문 ID가 없을 시 "Order Not Found" 반환
    """
    # 디버깅용
    print(f"get_order_status_from_db 호출됨: {order_id} ---")
    try:
        conn = sqlite3.connect(base_path + 'orders.db') # DB에 연결
        c = conn.cursor()                   # 커서: SQL 실행 도구
        # 오직 'status'만 조회
        c.execute("SELECT status FROM orders WHERE order_id = ?", (order_id,))
        result = c.fetchone() # (예: ('Delivered',))
        conn.close()
        
        if result:
            return result[0] # (예: 'Delivered')
        else:
            return "Order Not Found"
    except Exception as e:
        return f"DB Error: {str(e)}"

3. 고객의 배송 지연에 대한 대응 가능한 tool 정의 (배송 DB 검색 + 하드코딩) 
    - 지금은 쿠폰을 생성하고 등록 하는 등의 복잡한 DB 구성은 하드코딩으로만 구현

### 주의. 도구에서 도구 부르기
- tool 내에서 다른 tool (retriver 포함)을 호출하고자 한다면, 직접 적으로 호출() 하는 것이 아니라,
- invoke 형식으로 작성하여야만 함.

In [ ]:
# 3. issue_complaint_coupon 함수
@tool
def issue_complaint_coupon(order_id: str):
    """(DB 조회 기반) 
        'Shipping Delayed' 상태일 때만 쿠폰을 발급.
        단, 쿠폰 발급 외의 복잡한 로직은 하드코딩으로 처리.
        예를 들어, 쿠폰을 생성하기 위해서는 실제로 쿠폰 테이블에 삽입하거나,
        외부 시스템과 연동하는 등의 작업이 필요하지만, 여기서는 단순히
        주문 상태를 조회하고 조건에 맞으면 쿠폰 발급 메시지를 반환하는 것으로 대체
    """
    # 디버깅용
    print(f"issue_complaint_coupon 호출됨: {order_id} ---")
    order_status = get_order_status_from_db.invoke(order_id)

    # 만약, 배송 지연 상태라면 쿠폰 발급
    if order_status == "Shipping Delayed":
        return f"주문 ID {order_id}에 대해 5,000원 할인 쿠폰이 발급되었습니다."
    else:
        return f"주문 ID {order_id}는 쿠폰 발급 대상이 아닙니다. 현재 상태: {order_status}"

# 2. ReAct Agent 설계 및 실행

- 생각(Reason)하고 행동(Act)하는 Agent를 설꼐하고, 실제 작동 과정을 분석

## 2-1. ReAct (Reason + Act)란?

- LLM은 **추론(Reasonin)**과 **행동(Acting)**을 반복하며 복잡한 문제를 해결하는 프에임워크
1. **Thought (생각)**: "고객이 주문 상태를 묻네. `Get Order Status` 도구를 써야겠다."
2. **Action (행동)**: `Get Order Status[ORDER123]` 도구를 *실행*.
3. **Observation (관찰)**: "도구 결과: 'Delivered'"라는 *결과*를 받음.
4. **Thought (생각)**: "결과가 'Delivered'구나. 이제 고객에게 이걸 알려주자."

## 2-2. 시스템 프롬프트 설계

- Agent의 `규칙`을 정의하는 중요한 단계
1. Agent가 T-A-O 사이클을 따르게 하려면, **시스템 프롬프트**로 명확하게 지시해야 함
    - Thought - Action - Observation
2. Trustworthiness 규칙('쿠폰은 배송 지연 시에만 발급')을 이곳에서 명시할 것.
    - Agent가 확률에 의거하여 답변을 생성해서는 안됨
    - 정확히 설계된 도구에 의해 답변을 결정 할 수 있도록 지정

In [ ]:
system_prompt = """
당신은 AI 온라인 서점의 고객 서비스 AI 에이전트입니다.

[중요 규칙] 각 도구의 사용 조건과 설명을 반드시 따르세요.
* 'Issue_Complaint_Coupon' 도구는 'Get_Order_Status' 결과가 'Shipping Delayed'일 때만 사용.
* 주문 상태가 불명확하면, 쿠폰을 발급하기 전에 반드시 'Get_Order_Status'를 먼저 사용.
"""


## 2-3. Agent 생성 및 실행

- LLM, Tools, Prompt를 합쳐 Agent를 만드는 단계
1. **`create_agent`**
    - LLM, Tools, Prompt를 결합하는 함수
    - 내부적으로 T-A-O 루프를 구성하고, 도구 실행 결과를 적절한 값에 저장

In [ ]:
from langchain.agents import create_agent
from langchain_upstage import ChatUpstage

agent_tools = [retriever_tool, get_order_status_from_db, issue_complaint_coupon]

# LLM
llm = ChatUpstage() 
# Tools - agent_tools 변수
# Prompt - 위에서 만든 system_prompt 변수

# 1. ReAct Agent 생성
agent = create_agent(
    model=llm, 
    tools=agent_tools, 
    system_prompt=system_prompt)
print("ReAct Agent 생성 완료")

## 2-4. Agent 테스트 및 로그 분석

- `streaming`
    - LangChain은 실시간 업데이트를 표면화하기 위해 스트리밍 시스템을 구현
    - 완전한 응답이 준비되기 전에도 출력을 점진적으로 표시함으로써 스트리밍은 사용자 경험(UX)을 크게 향상시키며, 특히 LLM의 지연 시간을 처리
- 왜 그 답이 나왔는지 T-A-O 과정을 추적

### 2-4-1. 시나리오1: 배송 지연 + 쿠폰 요청 (Trustworthiness 테스트)
1. Thought
    - 모델이 주문 ID ORDER456의 배송 상태를 확인하고, 지연 시 쿠폰 발급 여부를 판단해야 한다고 결정함.
2. Action 1 — get_order_status_from_db
    - 입력: {"order_id": "ORDER456"}
    - 결과(Observation): "Shipping Delayed"
3. Thought
    - 배송 상태가 "Shipping Delayed"이므로 보상 쿠폰을 발급해야 한다고 판단함.
4. Action 2 — issue_complaint_coupon
    - 입력: {"order_id": "ORDER456"}
    - 내부 동작: get_order_status_from_db 재호출 후 상태 확인
    - 결과(Observation): "주문 ID ORDER456에 대해 5,000원 할인 쿠폰이 발급되었습니다."
5. 최종 응답
    - 주문 ID ORDER456의 배송 상태는 'Shipping Delayed'로 확인되었습니다. 이에 따라 5,000원 할인 쿠폰이 발급되었습니다.


In [ ]:
query_1 = "주문 ID ORDER456의 배송 상태가 어떻게 되나요? 만약 지연되었다면 쿠폰을 받을 수 있나요?"

for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": query_1}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print(data['messages'][0])

### 2-4-2. 시나리오 2: 배송 완료 + 쿠폰 요청 (Trustworthiness 거부 테스트)
- 실행 결과 직접 확인하기!

In [ ]:
query_2 = "제 주문 ORDER123인데요. 쿠폰 좀 발급해주세요."

for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": query_2}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print(data['messages'][0])

# Extra. Short-term Memory
## E-1. 들어가기 전에
- checkpoint를 제대로 활용하기 위해서는 langgraph를 활용할 수 있어야 함.
- 이번 챕터에서는 langgraph build에 대한 이야기는 잠시 접어두고, sqlite3를 사용해 대화를 기록하는 방법에 대해서만 간단히 다뤄볼 것
- langgraph 의존성 문제로 restart를 진행해야 할 수 있음.

In [34]:
!pip install langgraph-checkpoint-sqlite

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/bitsandbytes-0.45.4.dev0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.12.0.dev0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.0.dev0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: 

# E-2. agent 재 정의
### E-2-1. checkpointer
- 각 대화의 내용을 저장하고 관리 할 checkpointer를 정의
- `SqliteSaver`
    - sqlite3를 database로 하는 checkpointer를 설정
    - 단, SQLite는 기본적으로 생성된 스레드에서만 연결을 사용할 수 있도록 제한하는데, 이를 해제해야 함
    - check_same_thread=False 파라미터를 추가

- `AgentState`
    - 대화가 저장될 schema를 정의

In [ ]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

checkpointer = SqliteSaver(sqlite3.connect(base_path + "database.db", check_same_thread=False))

class CustomAgentState(AgentState):  
    user_id: str
    preferences: dict

agent = create_agent(
    model=llm, 
    tools=agent_tools, 
    system_prompt=system_prompt,
    state_schema=CustomAgentState,  
    checkpointer=checkpointer
  )

### E-2-2. RunnableConfig 설정
- 그래프 실행 설정을 담는 딕셔너리
- `config`: 대화 내용을 어디에 저장할 지 등의 설정을 진행
  - `thread_id`: 대화 세션을 식별하는 고유 ID
- `pretty_print`: 대화 내용중, 최종 답변 부분만 pretty 하게 출력

In [38]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [41]:
result = agent.invoke({"messages": [{"role": "user", "content": '배송 조회 하려면 뭐가 필요해?'}]}, config)
result['messages'][-1].pretty_print()

get_order_status_from_db 호출됨: order-123 ---
================================== Ai Message ==================================

주문 번호 order-123의 배송 상태를 조회하려고 했지만, 해당 주문은 데이터베이스에서 찾을 수 없습니다. 다른 주문 번호로 다시 시도하시거나, 다른 도움이 필요하시면 말씀해주세요.


In [42]:
result = agent.invoke({"messages": [{"role": "user", "content": '주문 ID ORDER456 이래'}]}, config)
result['messages'][-1].pretty_print()

get_order_status_from_db 호출됨: ORDER456 ---
issue_complaint_coupon 호출됨: ORDER456 ---
get_order_status_from_db 호출됨: ORDER456 ---
================================== Ai Message ==================================

주문 ID ORDER456의 배송 상태가 'Shipping Delayed'로 확인되었습니다. 이에 따라 5,000원 할인 쿠폰이 발급되었습니다. 쿠폰은 주문 시 사용하실 수 있습니다. 더 도움이 필요하시면 언제든지 말씀해주세요.


In [44]:
result = agent.invoke({"messages": [{"role": "user", "content": '뭐야 쿠폰 왜 주는거야?'}]}, config)
result['messages'][-1].pretty_print()

================================== Ai Message ==================================

AI 온라인 서점의 배송 정책에 따르면, 배송 지연 시 보상 정책은 다음과 같습니다:

- 평일 오후 3시 이전 주문 시 당일 발송됩니다.
- 오후 3시 이후 주문 건은 익일 발송됩니다.
- 주말 및 공휴일은 배송이 어렵습니다.

배송 지연 시 보상 정책은 주문 번호 order-123의 배송 상태를 마이페이지에서 조회할 수 있으며, 회원 및 비회원 모두 주문 번호로 배송 조회가 가능합니다. 배송 관련 문의는 고객센터로 연락 주시기 바랍니다.

더 자세한 정보가 필요하시면 언제든지 말씀해주세요.


In [46]:
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

배송 조회 하려면 뭐가 필요해?
================================== Ai Message ==================================
Tool Calls:
  Policy_Search (2419eb12-dc74-45a3-9f72-17196785baaf)
 Call ID: 2419eb12-dc74-45a3-9f72-17196785baaf
  Args:
    query: 배송 조회
================================= Tool Message =================================
Name: Policy_Search

## 배송 조회
- 주문 번호 order-123의 배송 상태는 마이페이지에서 조회 가능합니다.
- 회원 및 비회원 모두 주문 번호로 배송 조회가 가능합니다.
- 배송 관련 문의는 고객센터로 연락 주시기 바랍니다.

# AI 온라인 서점 배송 정책

## 일반 배송
- 평일 오후 3시 이전 주문 시 당일 발송됩니다.
- 오후 3시 이후 주문 건은 익일 발송됩니다.
- 주말 및 공휴일은 배송이 어렵습니다.

## 도서 산간 지역 배송
- 제주 및 도서 산간 지역은 추가 배송비가 발생할 수 있습니다.
- 추가 배송비 및 예상 소요 시간은 주문 시 확인 가능합니다.
================================== Ai Message ==================================

AI 온라인 서점의 배송 정책은 다음과 같습니다:

- 주문 번호 order-123의 배송 상태는 마이페이지에서 조회 가능합니다.
- 회원 및 비회원 모두 주문 번호로 배송 조회가 가능합니다.
- 배송 관련 문의는 고객센터로 연락 주시기 바랍니다.

일반 배송:
- 평일 오후 3시 이전 주문 시 당일 발송됩니다.
- 오후